# Virtual Cell Challenge — Week 1 Teacher Notebook
# Virtual Cell Challenge——第一周教师示范

**Question → code → interpretation / 问题 → 代码 → 自然语言解释**

This notebook uses synthetic teaching data, not official challenge data. 本 notebook 使用合成教学数据，不是官方比赛数据。

## 0. Reproducibility / 可复现性
Record the Python environment and use one fixed seed. 记录 Python 环境并固定随机种子。

In [ ]:
from pathlib import Path
import platform
import sys
COURSE_ROOT = Path.cwd() if (Path.cwd() / 'scripts').exists() else Path.cwd().parent
sys.path.insert(0, str(COURSE_ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import anndata as ad
from scipy import sparse

from scripts.make_toy_perturbseq import build_dataset
from scripts.pseudobulk import aggregate_pseudobulk

SEED = 2026
np.random.seed(SEED)
print('Python:', platform.python_version(), '| AnnData:', ad.__version__, '| seed:', SEED)

## 1. Create and load the dataset / 生成并读取数据

**Data contract / 数据契约**

- `X` and `layers['counts']`: raw non-negative integer counts / 原始非负整数 counts
- `obs`: cell-level design and QC metadata / 细胞层实验设计和 QC
- `var`: gene annotations / 基因注释
- `uns['ground_truth_log2fc']`: teaching truth only / 仅用于教学的真实效应

In [ ]:
data_path = COURSE_ROOT / 'data/week01_toy_perturbseq.h5ad'
if not data_path.exists():
    data_path.parent.mkdir(parents=True, exist_ok=True)
    build_dataset(seed=SEED).write_h5ad(data_path, compression='gzip')
adata = ad.read_h5ad(data_path)
adata

## 2. Audit AnnData without densifying / 不 densify 地审计 AnnData

`X` has no universal meaning. We infer its meaning from provenance and checks. `X` 没有固定语义，必须结合来源和数值检查。

In [ ]:
X = adata.X
values = X.data if sparse.issparse(X) else np.asarray(X).ravel()
audit = {
    'cells': adata.n_obs, 'genes': adata.n_vars,
    'matrix_type': type(X).__name__, 'dtype': str(X.dtype),
    'sparse': sparse.issparse(X), 'nnz': int(X.nnz),
    'nonnegative': bool((values >= 0).all()),
    'integer_like': bool(np.allclose(values, np.rint(values))),
    'obs_columns': list(adata.obs.columns),
    'var_columns': list(adata.var.columns),
    'layers': list(adata.layers.keys()),
}
pd.Series(audit)

In [ ]:
adata.obs[['target_gene', 'cell_type', 'donor', 'batch', 'guide_id']].head()

**Interpretation / 解释：** Cells are observational units. Donors define the independent replicate structure in this toy design. 细胞是观测单位；在这个 toy 设计中，donor 定义独立重复结构。

## 3. Sparse-safe QC / 稀疏安全的 QC
Compute library size, detected genes, and mitochondrial fraction without calling `.toarray()` on the full matrix.

In [ ]:
library_size = np.asarray(X.sum(axis=1)).ravel()
detected_genes = X.getnnz(axis=1)
mito_mask = np.asarray(adata.var['is_mito'], dtype=bool)
mito_counts = np.asarray(X[:, mito_mask].sum(axis=1)).ravel()
pct_mito = np.divide(mito_counts, library_size, out=np.zeros_like(mito_counts, dtype=float), where=library_size > 0) * 100
np.testing.assert_allclose(pct_mito, adata.obs['pct_mito'])
qc = adata.obs.assign(library_size_calc=library_size, detected_genes_calc=detected_genes, pct_mito_calc=pct_mito)
qc.groupby(['cell_type', 'target_gene'], observed=True)[['library_size_calc', 'detected_genes_calc', 'pct_mito_calc']].median().round(2)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))
axes[0].hist(library_size, bins=50); axes[0].set_title('Library size')
axes[1].hist(detected_genes, bins=50); axes[1].set_title('Detected genes')
axes[2].hist(pct_mito, bins=50); axes[2].set_title('Mitochondrial %')
for ax in axes: ax.set_ylabel('Cells')
plt.tight_layout()

**Teacher note / 教师提示：** Describe distributions before choosing thresholds. High mitochondrial fraction is evidence to investigate, not an automatic verdict. 先描述分布，再选择阈值；高线粒体比例是需要调查的证据，不是自动判决。

In [ ]:
library_floor = np.quantile(library_size, 0.01)
keep = (library_size >= library_floor) & (pct_mito <= 15)
filter_report = pd.Series({'before': adata.n_obs, 'after': int(keep.sum()), 'removed': int((~keep).sum()), 'library_floor': float(library_floor)})
filter_report

## 4. Pseudobulk at the replicate level / 按重复层级做 Pseudobulk

Sum raw counts by donor × cell type × condition, then prove count conservation. 按 donor × cell type × condition 对 raw counts 求和，并证明 counts 守恒。

In [ ]:
pb = aggregate_pseudobulk(adata, ['donor', 'cell_type', 'target_gene'], counts_layer='counts')
assert pb.n_obs == 3 * 2 * 4
np.testing.assert_array_equal(np.asarray(adata.layers['counts'].sum(axis=0)).ravel(), np.asarray(pb.X.sum(axis=0)).ravel())
pb.obs.head(8)

In [ ]:
def pb_mean(cell_type, condition):
    mask = np.asarray((pb.obs['cell_type'] == cell_type) & (pb.obs['target_gene'] == condition))
    return np.asarray(pb.X[mask].mean(axis=0)).ravel()

for cell_type in ['type_alpha', 'type_beta']:
    control = pb_mean(cell_type, 'non-targeting')
    pert = pb_mean(cell_type, 'GENE_A')
    log2fc = np.log2((pert + 1) / (control + 1))
    print(cell_type, 'GENE_A LFC=', round(log2fc[adata.var_names.get_loc('GENE_A')], 2), 'PATH_A_01 LFC=', round(log2fc[adata.var_names.get_loc('PATH_A_01')], 2))

## 5. Leakage-aware validation / 防泄漏验证

A valid first split is leave-one-donor-out. Randomly splitting cells would place near-related observations from one donor on both sides. 一个有效起点是 leave-one-donor-out；随机拆细胞会让同一 donor 的相关观测同时出现在训练和验证。

**Known limitation / 已知限制：** donor_3 is perfectly confounded with batch_2, so this toy design cannot identify donor and batch effects separately.

In [ ]:
train = adata[np.asarray(adata.obs['donor'] != 'donor_3')].copy()
valid = adata[np.asarray(adata.obs['donor'] == 'donor_3')].copy()
print('train:', train.shape, '| valid:', valid.shape)
assert set(train.obs['donor']).isdisjoint(set(valid.obs['donor']))

## Reflection / 反思

1. Where are raw counts stored, and what evidence supports that claim? / Raw counts 在哪里，证据是什么？
2. What is the independent replicate? / 独立重复是什么？
3. What does pseudobulk gain and lose? / Pseudobulk 得到什么、失去什么？
4. Which toy assumptions may fail in the real challenge? / 哪些 toy 假设在真实比赛中可能失效？